# 🧪 Practice Lab: Python Memory Management

**Netsetos GenAI Course** | Module 0 · Lesson 0.3 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+ (Google Colab recommended). All modules used (sys, gc, weakref, tracemalloc, threading) are in the stdlib.

### 🎯 You will:
1. Measure CPython object sizes and understand the PyMalloc allocator
2. Track reference counts through assignments, containers, and function calls
3. Create circular references and prove only GC can reclaim them
4. Benchmark the GIL's impact on threading vs multiprocessing
5. Build production AI memory patterns (weakref cache, gc.disable)
6. Hunt memory leaks using tracemalloc snapshot diffing

---

## Exercise 1 (Easy): CPython Object Sizes & PyMalloc

**📖 Concept:** Every Python object carries overhead: refcount (8 bytes) + type pointer (8 bytes) minimum. An empty `dict` costs 64 bytes. PyMalloc handles objects ≤512 bytes via arena allocator; larger objects use raw malloc. NumPy/PyTorch bypass PyMalloc entirely.

**📝 Task:**
1. Use `sys.getsizeof()` to measure 14+ object types
2. Print a sorted table showing bytes and allocator (PyMalloc ≤512 vs malloc >512)
3. Compute Python vs C overhead (Python int(0)=28 bytes, C int=4 bytes → 7x!)

**📤 Expected Output:**
```
  None                   16 bytes  [PyMalloc]
  int(0)                 28 bytes  [PyMalloc]
  ...
  list(1000)           8056 bytes  [malloc]
  Python int(0): 28 bytes vs C int: 4 bytes → 7x overhead
```

In [ ]:
# ✏️ YOUR CODE HERE
import sys
import numpy as np

objects = {
    "int(0)":          0,
    "int(2**30)":      2**30,
    "float(3.14)":     3.14,
    "bool(True)":      True,
    "None":            None,
    'str("")':         "",
    'str("hello")':    "hello",
    "list([])":        [],
    "list(1000)":      list(range(1000)),
    "dict({})":        {},
    "dict(100)":       {i: i for i in range(100)},
    "set()":           set(),
    "tuple()":         (),
    "np.zeros(1000)":  np.zeros(1000),
}

# TODO: print sorted table: name, size, PyMalloc vs malloc
# TODO: compute Python vs C overhead

<details><summary>💡 Hint 1 — Conceptual</summary>

`sys.getsizeof()` returns the shallow size — it doesn't include the elements inside a list/dict. For containers, the size is just the container overhead + pointers. Objects ≤512 bytes go through PyMalloc; larger ones use raw malloc.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
for name, obj in sorted(objects.items(), key=lambda x: sys.getsizeof(x[1])):
    size = sys.getsizeof(obj)
    allocator = "PyMalloc" if size <= 512 else "malloc"
    print(f'  {name:<22} {size:>6} bytes  [{allocator}]')
```

</details>

**✅ Success Criteria:**
- All 14 objects measured with sys.getsizeof()
- Table sorted by size with allocator classification
- Python vs C overhead computed

<details><summary>✅ Solution</summary>



In [ ]:
import sys
import numpy as np

objects = {
    "int(0)":          0,
    "int(2**30)":      2**30,
    "float(3.14)":     3.14,
    "bool(True)":      True,
    "None":            None,
    'str("")':         "",
    'str("hello")':    "hello",
    "list([])":        [],
    "list(1000)":      list(range(1000)),
    "dict({})":        {},
    "dict(100)":       {i: i for i in range(100)},
    "set()":           set(),
    "tuple()":         (),
    "np.zeros(1000)":  np.zeros(1000),
}

print(f'{"Object":<22} {"Size (bytes)":>12}  {"Allocator":<10}')
print('-' * 50)
for name, obj in sorted(objects.items(), key=lambda x: sys.getsizeof(x[1])):
    size = sys.getsizeof(obj)
    allocator = "PyMalloc" if size <= 512 else "malloc"
    print(f'  {name:<20} {size:>10}  [{allocator}]')

print(f'\n--- Python vs C Overhead ---')
print(f'  Python int(0): {sys.getsizeof(0)} bytes vs C int: 4 bytes → {sys.getsizeof(0)/4:.0f}x overhead')
print(f'  Python float:  {sys.getsizeof(3.14)} bytes vs C double: 8 bytes → {sys.getsizeof(3.14)/8:.1f}x overhead')

</details>

---

## Exercise 2 (Easy): Reference Counting Explorer

**📖 Concept:** Every PyObject has `ob_refcnt`. `b = a` increments it, `del b` decrements it. When refcount hits 0 → freed immediately. `sys.getrefcount()` always reports +1 (passing obj as arg creates a temp ref). Small integers (-5 to 256) are cached singletons.

**📝 Task:**
1. Track refcount through: creation, alias, list append, del, list remove
2. Show integer caching (refcount of 1 vs 257)
3. Prove `__del__` fires immediately when refcount → 0

**📤 Expected Output:**
```
1. After creation:     2
2. After b = a:        3
3. After list append:  4
4. After del b:        3
5. After list remove:  2
refcount(1):   ~1000+  (singleton)
refcount(257): 3       (not cached)
  → test freed (refcount hit 0)
```

In [ ]:
# ✏️ YOUR CODE HERE
import sys

# Part A: Track refcounts
a = [1, 2, 3]
print(f"1. After creation:     {sys.getrefcount(a)}")

b = a
print(f"2. After b = a:        {sys.getrefcount(a)}")

# TODO: add to container, del b, remove from container, print each

# Part B: Integer caching
# TODO: compare refcount(1) vs refcount(257)

# Part C: __del__ fires immediately
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"  → {self.name} freed (refcount hit 0)")

# TODO: create and del a Tracked object

<details><summary>💡 Hint 1 — Conceptual</summary>

`sys.getrefcount(obj)` always reports +1 because passing `obj` as an argument creates a temporary reference. So a freshly created variable shows 2, not 1.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
container = [a]  # +1 ref
print(f"3. list append: {sys.getrefcount(a)}")  # 4
del b
print(f"4. del b:       {sys.getrefcount(a)}")  # 3
container.pop()
print(f"5. list remove: {sys.getrefcount(a)}")  # 2
```

</details>

**✅ Success Criteria:**
- Refcount tracked correctly through all 5 operations
- Integer caching demonstrated (1 vs 257)
- __del__ fires immediately on refcount → 0

<details><summary>✅ Solution</summary>



In [ ]:
import sys

# Part A: Track refcounts
print('--- Part A: Reference Count Tracking ---')
a = [1, 2, 3]
print(f"1. After creation:     {sys.getrefcount(a)}")

b = a
print(f"2. After b = a:        {sys.getrefcount(a)}")

container = [a]
print(f"3. After list append:  {sys.getrefcount(a)}")

del b
print(f"4. After del b:        {sys.getrefcount(a)}")

container.pop()
print(f"5. After list remove:  {sys.getrefcount(a)}")

# Part B: Integer caching
print('\n--- Part B: Integer Caching ---')
print(f"refcount(1):   {sys.getrefcount(1):>6}  (cached singleton)")
print(f"refcount(42):  {sys.getrefcount(42):>6}  (also cached)")
print(f"refcount(257): {sys.getrefcount(257):>6}  (NOT cached)")
print(f"refcount(999): {sys.getrefcount(999):>6}  (NOT cached)")

# Part C: __del__ fires immediately
print('\n--- Part C: Immediate __del__ ---')
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"  → {self.name} freed (refcount hit 0)")

obj = Tracked("test-object")
print("Deleting...")
del obj
print("Done — __del__ already fired above (deterministic!)")

</details>

---

## Exercise 3 (Medium): Circular Reference Detector & Generational GC

**📖 Concept:** Refcounting can't detect cycles (A→B→A). The generational GC tracks container objects across 3 generations with thresholds `(700, 10, 10)`. Gen 0 is collected most often (most objects die young).

**📝 Task:**
1. Create A→B→A cycle. Delete both. Prove `__del__` does NOT fire.
2. `gc.collect()` breaks the cycle and triggers `__del__`
3. Print `gc.get_threshold()` and `gc.get_count()`
4. Create cycles in a loop, collect per-generation

**📤 Expected Output:**
```
After del (nothing freed — cycle keeps refcount > 0)
  → A collected by GC
  → B collected by GC
Thresholds: (700, 10, 10)
```

In [ ]:
# ✏️ YOUR CODE HERE
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"  → {self.name} collected by GC")

# Create a cycle
a = Node("A")
b = Node("B")
a.partner = b  # A → B
b.partner = a  # B → A (cycle!)

print("Before del:")
del a, b
print("After del (nothing freed — cycle)")

# TODO: gc.collect() to break the cycle
# TODO: print gc.get_threshold() and gc.get_count()
# TODO: per-generation collection

<details><summary>💡 Hint 1 — Conceptual</summary>

When you `del a, b`, each object's refcount drops from 2 to 1 (the partner still references it). Since neither hits 0, `__del__` never fires. The GC finds the cycle by tracing all container objects and detecting unreachable reference loops.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
collected = gc.collect()
print(f"GC collected {collected} objects")
print(f"Thresholds: {gc.get_threshold()}")
print(f"Counts:     {gc.get_count()}")
```

</details>

**✅ Success Criteria:**
- Circular reference created (A→B→A)
- del does NOT trigger __del__
- gc.collect() breaks cycle and triggers __del__
- GC thresholds and counts printed
- Per-generation collection demonstrated

<details><summary>✅ Solution</summary>



In [ ]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"  → {self.name} collected by GC")

print('--- Part A: Circular Reference ---')
gc.collect()  # Clean slate

a = Node("A")
b = Node("B")
a.partner = b
b.partner = a

print("Before del:")
del a, b
print("After del (nothing freed — cycle keeps refcount > 0)")

collected = gc.collect()
print(f"gc.collect() → freed {collected} objects\n")

print('--- Part B: GC State ---')
print(f"Thresholds: {gc.get_threshold()}")
print(f"Counts:     {gc.get_count()}\n")

print('--- Part C: Per-Generation Collection ---')

class SilentNode:
    def __init__(self):
        self.partner = None

for i in range(10):
    x = SilentNode()
    y = SilentNode()
    x.partner = y
    y.partner = x
    del x, y

print(f"Before collection — Counts: {gc.get_count()}")
collected_g0 = gc.collect(0)
print(f"gc.collect(0) [Gen 0]: freed {collected_g0}")
collected_g1 = gc.collect(1)
print(f"gc.collect(1) [Gen 0+1]: freed {collected_g1}")
collected_full = gc.collect(2)
print(f"gc.collect(2) [Full]: freed {collected_full}")

</details>

---

## Exercise 4 (Medium): GIL Impact Benchmark

**📖 Concept:** The GIL allows only one thread to execute Python bytecode at a time. Threading does NOT speed up CPU-bound Python. But for I/O-bound tasks (API calls, sleep), threading gives near-linear speedup because the GIL is released during I/O waits.

**📝 Task:**
1. Write a CPU-bound function (count to 5M in a Python loop)
2. Write a simulated I/O function (`time.sleep(1)`)
3. Run 4 copies: sequential vs threaded (4 threads). Time each.
4. Print comparison table showing where GIL blocks

**📤 Expected Output:**
```
CPU-bound: Sequential ~2.0s, Threaded ~2.1s (GIL blocks!)
I/O-bound: Sequential ~4.0s, Threaded ~1.0s (GIL released!)
```

In [ ]:
# ✏️ YOUR CODE HERE
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(n=5_000_000):
    """CPU-bound: pure Python loop."""
    total = 0
    for i in range(n):
        total += i
    return total

def io_work(seconds=1.0):
    """I/O-bound: simulated API call."""
    time.sleep(seconds)
    return "done"

NUM_TASKS = 4

# TODO: sequential CPU vs threaded CPU
# TODO: sequential I/O vs threaded I/O
# TODO: print comparison table

<details><summary>💡 Hint 1 — Conceptual</summary>

For CPU-bound: sequential ≈ threaded (GIL blocks). For I/O-bound: threaded ≈ 4x faster (GIL released during sleep). This is why LLM API calls use async/threading, not multiprocessing.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
start = time.time()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as ex:
    list(ex.map(lambda _: cpu_work(), range(NUM_TASKS)))
thr_cpu = time.time() - start
```

</details>

**✅ Success Criteria:**
- CPU sequential ≈ threaded (GIL proof)
- I/O threaded ~4x faster than sequential
- Comparison table printed with verdict

<details><summary>✅ Solution</summary>



In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(n=5_000_000):
    total = 0
    for i in range(n):
        total += i
    return total

def io_work(seconds=1.0):
    time.sleep(seconds)
    return "done"

NUM_TASKS = 4

# CPU-bound
print('--- CPU-bound (count to 5M × 4) ---')
start = time.time()
for _ in range(NUM_TASKS):
    cpu_work()
seq_cpu = time.time() - start
print(f'  Sequential:   {seq_cpu:.2f}s')

start = time.time()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as ex:
    list(ex.map(lambda _: cpu_work(), range(NUM_TASKS)))
thr_cpu = time.time() - start
print(f'  Threaded (4): {thr_cpu:.2f}s')
print(f'  Speedup:      {seq_cpu/thr_cpu:.2f}x  (≈1x → GIL blocks!)\n')

# I/O-bound
print('--- I/O-bound (sleep 1s × 4) ---')
start = time.time()
for _ in range(NUM_TASKS):
    io_work()
seq_io = time.time() - start
print(f'  Sequential:   {seq_io:.2f}s')

start = time.time()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as ex:
    list(ex.map(lambda _: io_work(), range(NUM_TASKS)))
thr_io = time.time() - start
print(f'  Threaded (4): {thr_io:.2f}s')
print(f'  Speedup:      {seq_io/thr_io:.2f}x  (≈4x → GIL released!)\n')

print('--- Verdict ---')
print(f'  CPU-bound: {seq_cpu/thr_cpu:.1f}x → use multiprocessing')
print(f'  I/O-bound: {seq_io/thr_io:.1f}x → use threading/async')

</details>

---

## Exercise 5 (Hard): Production AI Memory Patterns

**📖 Concept:** Three production patterns: (1) `weakref.WeakValueDictionary` for auto-evicting embedding caches, (2) `gc.disable()` during inference to prevent random GC pauses, (3) explicit `del + gc.collect()` between epochs.

**📝 Task:**
1. Build a WeakRefEmbeddingCache — entries auto-evict when strong refs are deleted
2. Compare vs regular dict (grows forever)
3. Implement gc.disable/enable pattern with timing
4. Simulate epoch cleanup loop with memory tracking

**📤 Expected Output:**
```
After storing 5: size=5
After gc.collect: size=2  (weakrefs auto-evicted!)
Epoch 1: allocated 5MB → cleanup → freed
```

In [ ]:
# ✏️ YOUR CODE HERE
import gc, weakref, time
import numpy as np

class Embedding:
    def __init__(self, text):
        self.text = text
        self.vector = np.random.randn(1536).astype(np.float32)

class WeakRefEmbeddingCache:
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()
        self.hits = 0
        self.misses = 0

    def get(self, text):
        # TODO: check cache → hit or miss, return embedding
        pass

    def __len__(self):
        return len(self._cache)

# TODO: store 5 embeddings, keep strong refs to 2, del rest
# TODO: gc.collect(), show cache shrank
# TODO: gc.disable/enable pattern
# TODO: epoch cleanup loop

<details><summary>💡 Hint 1 — Conceptual</summary>

`WeakValueDictionary` holds weak references to values. When the last strong reference is deleted, the entry automatically disappears. This prevents unbounded memory growth in caches.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
def get(self, text):
    if text in self._cache:
        self.hits += 1
        return self._cache[text]
    self.misses += 1
    emb = Embedding(text)
    self._cache[text] = emb
    return emb
```

</details>

**✅ Success Criteria:**
- WeakRef cache stores/retrieves embeddings
- Auto-eviction proven when strong refs deleted
- gc.disable/enable pattern with timing
- Epoch cleanup loop with memory tracking

<details><summary>✅ Solution</summary>



In [ ]:
import gc, weakref, time, tracemalloc
import numpy as np

class Embedding:
    def __init__(self, text):
        self.text = text
        self.vector = np.random.randn(1536).astype(np.float32)

# --- Pattern 1: WeakRef Cache ---
print('--- Pattern 1: WeakRef Cache ---')

class WeakRefEmbeddingCache:
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()
        self.hits = 0
        self.misses = 0

    def get(self, text):
        if text in self._cache:
            self.hits += 1
            return self._cache[text]
        self.misses += 1
        emb = Embedding(text)
        self._cache[text] = emb
        return emb

    def stats(self):
        return f'size={len(self._cache)}, hits={self.hits}, misses={self.misses}'

cache = WeakRefEmbeddingCache()
keep1 = cache.get("hello")
keep2 = cache.get("world")
_ = cache.get("temp1")
_ = cache.get("temp2")
_ = cache.get("temp3")
_ = None

print(f'After storing 5: {cache.stats()}')
gc.collect()
print(f'After gc.collect: {cache.stats()}')
print('(temp entries auto-evicted!)\n')

# --- Pattern 2: gc.disable ---
print('--- Pattern 2: gc.disable During Inference ---')

def simulated_inference():
    for _ in range(100):
        a = np.random.randn(256, 256)
        c = a @ a.T
    return c

times_on, times_off = [], []
for _ in range(5):
    start = time.time()
    simulated_inference()
    times_on.append((time.time() - start) * 1000)

for _ in range(5):
    gc.disable()
    start = time.time()
    simulated_inference()
    times_off.append((time.time() - start) * 1000)
    gc.enable()
    gc.collect()

print(f'  GC enabled:  {sum(times_on)/5:.1f}ms avg')
print(f'  GC disabled: {sum(times_off)/5:.1f}ms avg\n')

# --- Pattern 3: Epoch Cleanup ---
print('--- Pattern 3: Epoch Cleanup ---')
tracemalloc.start()

for epoch in range(3):
    big_batch = np.random.randn(10000, 128).astype(np.float32)
    current, peak = tracemalloc.get_traced_memory()
    print(f'  Epoch {epoch+1}: allocated {current/1024/1024:.1f}MB')
    del big_batch
    gc.collect()
    current, _ = tracemalloc.get_traced_memory()
    print(f'           cleanup: {current/1024/1024:.1f}MB')

tracemalloc.stop()

</details>

---

## Exercise 6 (Hard): Memory Leak Hunter with tracemalloc

**📖 Concept:** `tracemalloc` tracks every allocation's source file and line number. Snapshot A → leaky code → Snapshot B → diff reveals exactly where memory grew. This is how you debug "why is my server using 12GB after 4 hours?"

**📝 Task:**
1. Start tracemalloc, take snapshot A
2. Run a function with an intentional leak (appending to a global list)
3. Take snapshot B, diff A→B — identify the leaking line
4. Fix the leak (clear list or bounded deque)
5. Take snapshot C, diff B→C — prove the fix
6. Print peak and current memory

**📤 Expected Output:**
```
Top memory growers (A → B):
  <cell>:10: size=3012 KiB (+3012 KiB)  ← THE LEAK
After fix:
  <cell>:10: size=0 B (-3012 KiB)  ← FIXED
```

In [ ]:
# ✏️ YOUR CODE HERE
import tracemalloc
import gc
import numpy as np

tracemalloc.start()
snap_a = tracemalloc.take_snapshot()

# Leaky function
leaked_cache = []

def process_batch(batch_id):
    embedding = np.random.randn(1536).astype(np.float32)
    leaked_cache.append(embedding)  # LEAK!
    return f"Batch {batch_id} done"

# Run 500 batches
for i in range(500):
    process_batch(i)

snap_b = tracemalloc.take_snapshot()

# TODO: diff snap_a → snap_b, print top 3
# TODO: fix the leak (clear or bounded deque)
# TODO: snap_c, diff snap_b → snap_c
# TODO: print current and peak memory

<details><summary>💡 Hint 1 — Conceptual</summary>

`snap_b.compare_to(snap_a, "lineno")` returns StatisticDiff objects sorted by size difference. The top entry shows where memory grew most — the `leaked_cache.append()` line.

</details>

<details><summary>💡 Hint 2 — Code</summary>

```python
print("Top memory growers:")
for stat in snap_b.compare_to(snap_a, "lineno")[:3]:
    print(f"  {stat}")

leaked_cache.clear()
gc.collect()
snap_c = tracemalloc.take_snapshot()
current, peak = tracemalloc.get_traced_memory()
```

</details>

**✅ Success Criteria:**
- tracemalloc snapshots at A, B, C
- Diff A→B identifies the leaking line
- Leak fixed, diff B→C proves it
- Peak and current memory printed

<details><summary>✅ Solution</summary>



In [ ]:
import tracemalloc
import gc
import numpy as np
from collections import deque

tracemalloc.start()
snap_a = tracemalloc.take_snapshot()

# --- Leaky code ---
leaked_cache = []

def process_batch(batch_id):
    embedding = np.random.randn(1536).astype(np.float32)
    leaked_cache.append(embedding)
    return f"Batch {batch_id} processed"

for i in range(500):
    process_batch(i)

snap_b = tracemalloc.take_snapshot()

print('=== LEAK DETECTION: A → B ===')
for stat in snap_b.compare_to(snap_a, 'lineno')[:3]:
    print(f'  {stat}')

current_b, peak_b = tracemalloc.get_traced_memory()
print(f'Current: {current_b/1024:.1f} KB, Peak: {peak_b/1024:.1f} KB')
print(f'Leaked cache: {len(leaked_cache)} entries\n')

# --- Fix ---
print('=== FIX ===')
leaked_cache.clear()
gc.collect()
snap_c = tracemalloc.take_snapshot()

print('Memory changes (B → C):')
for stat in snap_c.compare_to(snap_b, 'lineno')[:3]:
    print(f'  {stat}')

current_c, peak_c = tracemalloc.get_traced_memory()
print(f'Current: {current_c/1024:.1f} KB, Peak: {peak_c/1024:.1f} KB')
print(f'Freed: {(current_b - current_c)/1024:.1f} KB\n')

# --- Bonus: Bounded deque ---
print('=== BONUS: Bounded deque(maxlen=100) ===')
bounded = deque(maxlen=100)
for i in range(500):
    bounded.append(np.random.randn(1536).astype(np.float32))
print(f'After 500 inserts: {len(bounded)} entries (bounded!)')

tracemalloc.stop()

</details>

---

## 🎉 Done!

You've mastered all 6 Python memory management concepts for production AI:
- **CPython Architecture** — object sizes, PyMalloc vs malloc
- **Reference Counting** — deterministic, immediate, predictable
- **Circular References & GC** — generational collector, thresholds, per-gen collection
- **The GIL** — threading vs multiprocessing vs async
- **Production Patterns** — weakref caches, gc.disable, explicit cleanup
- **tracemalloc** — snapshot diffing to find and fix memory leaks

These are the exact skills tested in senior Python interviews — and the patterns you'll use daily in production AI deployments. Next: **Module 1.**